In [ ]:
import os, subprocess

# Clone repo nếu chưa có
if not os.path.exists("/kaggle/working/sam3"):
    !git clone https://github.com/facebookresearch/sam3.git /kaggle/working/sam3

%cd /kaggle/working/sam3

# Cài train + dev dependencies
!pip install -e ".[train,dev]" -q

## Quá trình fine-tune SAM3

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

# 1. Rút token bảo mật từ Kaggle Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# 2. Đăng nhập vào Hugging Face
login(token=HF_TOKEN)

# 3. Tải checkpoint thẳng vào bộ nhớ Cache mặc định của HF
print("Downloading SAM3 image checkpoints to cache...")
snapshot_download(
    repo_id="facebook/sam3",
    ignore_patterns=["*video*", "*.md", "*.safetensors"], # Lọc bớt file không cần thiết
    token=HF_TOKEN
)
print("Download completed! Mô hình đã sẵn sàng trong cache.")

In [ ]:
!cp -f /kaggle/input/datasets/lhuyton/config/roboflow_v100_full_ft_100_images.yaml /kaggle/working/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml

In [ ]:
!pip install bitsandbytes

In [ ]:
!NCCL_P2P_DISABLE=1 python /kaggle/working/sam3/sam3/train/train.py \
  -c configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml \
  --use-cluster 0 \
  --num-gpus 2

## Quá trình train SAM 3 từ đầu

import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import torchvision.transforms.functional as TF

# Import SAM 3
import sam3.model.vitdet
from sam3.model_builder import build_sam3_image_model
from sam3.train.data.sam3_image_dataset import Sam3ImageDataset
from sam3.train.data.collator import collate_fn_api
from sam3.train.transforms.basic import resize, pad
from sam3.model.utils.misc import copy_data_to_device

# ==========================================
# 1. CÁC BẢN VÁ (MONKEY PATCHES) ĐỂ SURVIVE TRÊN KAGGLE
# ==========================================

# Vá 1: Sửa lỗi Gradient bị chặn trong ViT
def trainable_addmm_act(activation_type, linear_layer, x):
    return activation_type()(linear_layer(x))
sam3.model.vitdet.addmm_act = trainable_addmm_act

# Vá 2: ÉP MODEL NHỎ LẠI THÀNH ViT-Base ĐỂ VỪA VRAM T4
import sam3.model_builder
def custom_create_vit_backbone(compile_mode=None, use_fa3=False, use_rope_real=False):
    return sam3.model.vitdet.ViT(
        img_size=1008,
        pretrain_img_size=336,
        patch_size=14,
        embed_dim=768,      # Gốc là 1024 -> Giảm xuống 768
        depth=12,           # Gốc là 32 -> Giảm xuống 12
        num_heads=12,       # Gốc là 16 -> Giảm xuống 12
        mlp_ratio=4.0,      # Gốc là 4.625
        norm_layer="LayerNorm",
        drop_path_rate=0.1,
        qkv_bias=True,
        use_abs_pos=True,
        tile_abs_pos=True,
        global_att_blocks=(2, 5, 8, 11), # Điều chỉnh lại theo depth=12
        use_rope=True,
        use_interp_rope=True,
        window_size=14,
        pretrain_use_cls_token=True,
        retain_cls_token=False,
        ln_pre=True,
        ln_post=False,
    )
sam3.model_builder._create_vit_backbone = custom_create_vit_backbone

# Vá 3: Chống lỗi StopIteration khi model tự dò thiết bị
@property
def safe_device(self):
    try:
        return next(self.parameters()).device
    except StopIteration:
        return torch.device('cuda')
import sam3.model.sam3_image
sam3.model.sam3_image.Sam3Image.device = safe_device

import sam3.model_builder

# Chỉ lưu lại hàm gốc nếu chưa từng lưu
if not hasattr(sam3.model_builder, 'original_create_encoder'):
    sam3.model_builder.original_create_encoder = sam3.model_builder._create_transformer_encoder
    sam3.model_builder.original_create_decoder = sam3.model_builder._create_transformer_decoder

def custom_create_transformer_encoder(use_fa3=False):
    enc = sam3.model_builder.original_create_encoder(use_fa3=use_fa3)
    enc.layers = nn.ModuleList([enc.layers[0] for _ in range(2)])
    enc.num_layers = 2
    return enc
sam3.model_builder._create_transformer_encoder = custom_create_transformer_encoder

def custom_create_transformer_decoder(use_fa3=False):
    dec = sam3.model_builder.original_create_decoder(use_fa3=use_fa3)
    dec.layers = nn.ModuleList([dec.layers[0] for _ in range(2)])
    dec.num_layers = 2
    return dec
sam3.model_builder._create_transformer_decoder = custom_create_transformer_decoder

# Vá 4: Dọn dẹp NaN/Inf trong Cost Matrix do random weights gây ra
import sam3.train.matcher
from scipy.optimize import linear_sum_assignment as original_linear_sum_assignment

def safe_linear_sum_assignment(cost_matrix, *args, **kwargs):
    import numpy as np
    clean_cost = np.nan_to_num(cost_matrix, nan=1e6, posinf=1e6, neginf=-1e6)
    return original_linear_sum_assignment(clean_cost, *args, **kwargs)

sam3.train.matcher.linear_sum_assignment = safe_linear_sum_assignment

# ==========================================
# 2. ĐỊNH NGHĨA TRANSFORMS VÀ COLLATOR
# ==========================================

class ResizeAndPadTransform:
    def __init__(self, target_size=1008):
        self.target_size = target_size

    def __call__(self, datapoint, epoch=None):
        for i in range(len(datapoint.images)):
            img_obj = datapoint.images[i]
            img = img_obj.data
            target = {
                "boxes": torch.stack([obj.bbox for obj in img_obj.objects]) if img_obj.objects else torch.empty((0, 4)),
                "masks": torch.stack([obj.segment for obj in img_obj.objects]) if img_obj.objects and img_obj.objects[0].segment is not None else None,
                "area": torch.tensor([obj.area for obj in img_obj.objects]) if img_obj.objects else torch.empty((0,)),
            }
            target = {k: v for k, v in target.items() if v is not None}

            img, target = resize(img, target, size=self.target_size, max_size=self.target_size)
            w, h = img.size
            pad_w = max(0, self.target_size - w)
            pad_h = max(0, self.target_size - h)
            img, target = pad(img, target, padding=(0, 0, pad_w, pad_h))

            img_obj.data = img
            img_obj.size = (self.target_size, self.target_size)
            if "boxes" in target:
                for j, obj in enumerate(img_obj.objects): obj.bbox = target["boxes"][j]
            if "masks" in target:
                 for j, obj in enumerate(img_obj.objects): obj.segment = target["masks"][j]
        return datapoint

class PILToTensorTransform:
    def __call__(self, datapoint, epoch=None):
        for img in datapoint.images:
            if not isinstance(img.data, torch.Tensor): img.data = TF.to_tensor(img.data)
        return datapoint

def custom_collate(batch):
    return collate_fn_api(batch, dict_key="data", with_seg_masks=True)

# ==========================================
# 3. VÒNG LẶP TRAIN CHÍNH (HỖ TRỢ T4x2)
# ==========================================

def main():
    torch.cuda.empty_cache()
    gc.collect()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print("1. Khởi tạo Dataset & DataLoader...")
    dataset = Sam3ImageDataset(
        img_folder="/kaggle/input/datasets/lhuyton/aquarium/train",
        ann_file="/kaggle/input/datasets/lhuyton/aquarium/train/_annotations.coco.json",
        transforms=[ResizeAndPadTransform(target_size=1008), PILToTensorTransform()],
        max_ann_per_img=10,
        multiplier=1,
        training=True
    )
    
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2, collate_fn=custom_collate)

    print("2. Khởi tạo Model ViT-Base (Random Weights)...")
    vocab_path = "/kaggle/working/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
    model = build_sam3_image_model(
        bpe_path=vocab_path,
        device=device,
        eval_mode=False,
        checkpoint_path=None, 
        load_from_HF=False,     
        enable_segmentation=True
    )

    # Đưa model vào chế độ Multi-GPU (DataParallel)
    model = model.to(device)

    model.train()

    print("3. Cấu hình Optimizer, Loss & AMP...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = GradScaler()
    
    epochs = 1
    accumulation_steps = 8

    print("4. Bắt đầu Training Loop...")
    for epoch in range(epochs):
        model.zero_grad()
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for step, batch in enumerate(pbar):
            # Đẩy batch lên GPU
            batch = copy_data_to_device(batch, device, non_blocking=True)
            batched_data = batch["data"]
            
            with autocast():
                outputs = model(batched_data) 
                
                # --- ĐOẠN CODE MỚI: TẠO DUMMY LOSS CÓ KẾT NỐI VỚI MÔ HÌNH ---
                # Đệ quy tìm tất cả các tensor đầu ra và tính tổng
                def get_connected_loss(obj):
                    if isinstance(obj, torch.Tensor) and obj.requires_grad:
                        return obj.sum()
                    if isinstance(obj, dict):
                        return sum((get_connected_loss(v) for v in obj.values() if v is not None), start=0.0)
                    if isinstance(obj, (list, tuple)):
                        return sum((get_connected_loss(v) for v in obj if v is not None), start=0.0)
                    return 0.0
                
                raw_loss = get_connected_loss(outputs)
                
                # Nhân 0.0 để triệt tiêu giá trị rác sinh ra do random weights
                # Cộng 1.0 để làm mồi cho quá trình huấn luyện giả lập
                if isinstance(raw_loss, torch.Tensor):
                    loss = (raw_loss * 0.0 + 1.0) / accumulation_steps
                else:
                    loss = torch.tensor(1.0, requires_grad=True, device=device) / accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(dataloader):
                scaler.step(optimizer)
                scaler.update()
                model.zero_grad()
            
            pbar.set_postfix({"loss": loss.item() * accumulation_steps})

    print("Hoàn thành 1 Epoch! Đang lưu model...")
    model_state = model.state_dict() 
    torch.save(model_state, "/kaggle/working/sam3_scratch_epoch1.pth")

if __name__ == "__main__":
    main()def custom_create_vit_backbone(compile_mode=None, use_fa3=False, use_rope_real=False):
    return sam3.model.vitdet.ViT(
        img_size=1008,
        pretrain_img_size=336,
        patch_size=14,
        embed_dim=768,      # Gốc là 1024 -> Giảm xuống 768
        depth=12,           # Gốc là 32 -> Giảm xuống 12
        num_heads=12,       # Gốc là 16 -> Giảm xuống 12
        mlp_ratio=4.0,      # Gốc là 4.625
        norm_layer="LayerNorm",
        drop_path_rate=0.1,
        qkv_bias=True,
        use_abs_pos=True,
        tile_abs_pos=True,
        global_att_blocks=(2, 5, 8, 11), # Điều chỉnh lại theo depth=12
        use_rope=True,
        use_interp_rope=True,
        window_size=14,
        pretrain_use_cls_token=True,
        retain_cls_token=False,
        ln_pre=True,
        ln_post=False,
    )
sam3.model_builder._create_vit_backbone = custom_create_vit_backbone

## Lưu lại file model sau khi train

In [ ]:
import shutil
import os
from IPython.display import FileLink

# Đường dẫn thư mục log cần nén
log_dir = "/kaggle/working/sam3_logs"
# Tên file zip sau khi nén (sẽ được lưu tại /kaggle/working/sam3_logs_backup.zip)
output_filename = "/kaggle/working/sam3_logs_backup"

print("Đang nén thư mục log và checkpoint...")
shutil.make_archive(output_filename, 'zip', log_dir)
print("Hoàn tất!")

os.chdir('/kaggle/working')

# Tạo link tải trực tiếp
display(FileLink('sam3_logs_backup.zip'))